### 1) Reproducibility and Configuration
Sets the deterministic seed and loads all project configuration files.


In [7]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
os.chdir('..')
print("Working dir:", os.getcwd())
from src.utils.seed import set_seed
from src.models.train import load_all_configs

set_seed(42)
configs = load_all_configs()
dataset_cfg = configs['dataset_config']
model_cfg   = configs['model_config']
training_cfg = configs['training_config']

SEED = int(model_cfg['random_seed'])
print('Configs loaded:', list(configs.keys()))
print('Seed:', SEED)


Working dir: C:\Users
Configs loaded: ['dataset_config', 'model_config', 'training_config']
Seed: 42


### 2) Load Raw Framingham Data
Load raw (not pre-processed) Framingham so we have access to `cigsPerDay`
for the smoking_packyears derivation before feature alignment drops it.


In [4]:
from src.data.loader import load_framingham

framingham_raw = load_framingham()
print('Raw Framingham shape:', framingham_raw.shape)
print('Columns:', list(framingham_raw.columns))
framingham_raw.head()


Raw Framingham shape: (3658, 16)
Columns: ['male', 'age', 'education', 'currentSmoker', 'cigsPerDay', 'BPMeds', 'prevalentStroke', 'prevalentHyp', 'diabetes', 'totChol', 'sysBP', 'diaBP', 'BMI', 'heartRate', 'glucose', 'TenYearCHD']


,male,age,education,currentSmoker,cigsPerDay,BPMeds,prevalentStroke,prevalentHyp,diabetes,totChol,sysBP,diaBP,BMI,heartRate,glucose,TenYearCHD
0,1,39,4.0,0,0.0,0.0,0,0,0,195.0,106.0,70.0,26.97,80.0,77.0,0
1,0,46,2.0,0,0.0,0.0,0,0,0,250.0,121.0,81.0,28.73,95.0,76.0,0
2,1,48,1.0,1,20.0,0.0,0,0,0,245.0,127.5,80.0,25.34,75.0,70.0,0
3,0,61,3.0,1,30.0,0.0,0,1,0,225.0,150.0,95.0,28.58,65.0,103.0,1
4,0,46,3.0,1,23.0,0.0,0,0,0,285.0,130.0,84.0,23.10,85.0,85.0,0


### 3) Generate Synthetic LRS Component Columns
Framingham does not contain the five LRS lifestyle columns directly.
This cell derives `smoking_packyears` from `cigsPerDay` and synthetically
generates the remaining four with a fixed seed (fully documented in
`src/data/lrs_generator.py` and the paper Limitations section).
Columns are generated on the FULL DataFrame before any split to ensure
consistent distributions across train/cal/holdout.


In [9]:
from src.data.lrs_generator import generate_lrs_columns

framingham_with_lifestyle = generate_lrs_columns(framingham_raw, seed=SEED)

LRS_COLS = ['smoking_packyears', 'sedentary_hours', 'activity_mets',
            'sleep_regularity', 'alcohol_units']

print('LRS columns generated:')
print(framingham_with_lifestyle[LRS_COLS].describe().round(3))


LRS columns generated:
       smoking_packyears  sedentary_hours  activity_mets  sleep_regularity  \
count           3658.000         3658.000       3658.000          3658.000   
mean               4.513            7.976         25.203             1.196   
std                5.961            2.001         10.508             0.580   
min                0.000            2.000          5.000             0.200   
25%                0.000            6.645         17.838             0.786   
50%                0.000            8.020         25.128             1.174   
75%               10.000            9.295         32.107             1.592   
max               35.000           14.358         60.000             3.615   

       alcohol_units  
count       3658.000  
mean           8.210  
std            5.566  
min            0.000  
25%            3.940  
50%            7.864  
75%           11.880  
max           32.907  


### 4) Compute and Append Lifestyle Risk Score (LRS)
Now that the five component columns exist, compute the composite LRS
feature using the canonical `lrs.py` implementation.


In [10]:
from src.features.lrs import compute_lrs, append_lrs

lrs_series = compute_lrs(framingham_with_lifestyle)
framingham_with_lrs = append_lrs(framingham_with_lifestyle, lrs_series)

print('LRS appended. Shape:', framingham_with_lrs.shape)
print('LRS stats:', lrs_series.describe().round(4))


LRS appended. Shape: (3658, 22)
LRS stats: count    3658.0000
mean        0.3573
std         0.0840
min         0.0754
25%         0.2991
50%         0.3558
75%         0.4150
max         0.6810
Name: LRS, dtype: float64


### 5) Feature Alignment, Three-Way Split, and Scaling
Align Framingham to the shared clinical feature space, perform the
strict 60/20/20 stratified split, fit the scaler on training data only,
and save all split CSVs so `04_calibration.ipynb` can load them directly.


In [11]:
import joblib
import pandas as pd
from pathlib import Path

from src.data.loader import load_cleveland
from src.data.preprocessing import fit_scaler, scale, three_way_split
from src.features.feature_alignment import align_and_document

# Align Framingham (now with LRS) against Cleveland to get shared clinical cols
cleveland_raw = load_cleveland()
framingham_aligned, cleveland_aligned = align_and_document(
    framingham_with_lrs, cleveland_raw
)

# Manually re-append LRS after alignment (alignment drops non-shared cols)
framingham_aligned['LRS'] = framingham_with_lrs.loc[
    framingham_aligned.index, 'LRS'
].values

target_col = dataset_cfg['framingham_target_column']
feature_cols = [c for c in framingham_aligned.columns if c != target_col]

X = framingham_aligned[feature_cols]
y = framingham_aligned[target_col]

print('Feature columns used for training:', feature_cols)
print('X shape:', X.shape, '| y shape:', y.shape)
print('Class balance:', y.value_counts().to_dict())



+--------------------------------------------------------------+
|          FEATURE ALIGNMENT REPORT                            |
+--------------------------------------------------------------+
   dataset     original_name                                                                          reason
framingham         education Socio-demographic variable, not a clinical risk factor; no Cleveland equivalent
framingham     currentSmoker                   Binary smoking flag; no Cleveland equivalent. Captured by LRS
framingham        cigsPerDay            Continuous smoking measure; no Cleveland equivalent. Captured by LRS
framingham            BPMeds                         Blood-pressure medication flag; no Cleveland equivalent
framingham   prevalentStroke                                   Medical history flag; no Cleveland equivalent
framingham      prevalentHyp                                   Medical history flag; no Cleveland equivalent
framingham             diaBP             

In [20]:
X_train, X_cal, X_holdout, y_train, y_cal, y_holdout = three_way_split(X, y)

scaler = fit_scaler(X_train)
X_train_scaled  = scale(X_train, scaler)
X_cal_scaled    = scale(X_cal, scaler)
X_holdout_scaled = scale(X_holdout, scaler)

# --- Persist everything needed by 04_calibration.ipynb ---
models_dir = Path('models')
models_dir.mkdir(parents=True, exist_ok=True)
processed_dir = Path('data/processed')
processed_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(scaler, models_dir / 'scaler.pkl')

X_train_scaled.to_csv(processed_dir / 'X_train.csv', index=False)
X_cal_scaled.to_csv(processed_dir   / 'X_calib.csv', index=False)
X_holdout_scaled.to_csv(processed_dir / 'X_holdout.csv', index=False)
y_train.to_csv(processed_dir / 'y_train.csv', index=False)
y_cal.to_csv(processed_dir   / 'y_calib.csv', index=False)
y_holdout.to_csv(processed_dir / 'y_holdout.csv', index=False)

# Save feature column order — inference pipeline must match this exactly
import json
with open(models_dir / 'feature_columns.json', 'w') as f:
    json.dump(feature_cols, f, indent=2)

print('Split shapes — train:', X_train_scaled.shape,
      '| cal:', X_cal_scaled.shape,
      '| holdout:', X_holdout_scaled.shape)
print('Saved scaler, splits, and feature_columns.json to models/ and data/processed/')


Split shapes — train: (2194, 7) | cal: (732, 7) | holdout: (732, 7)
Saved scaler, splits, and feature_columns.json to models/ and data/processed/


### 6) Train XGBoost and Log to W&B


In [21]:
from src.models.train import train_xgboost

xgb_model = train_xgboost(X_train_scaled, y_train, X_cal_scaled, y_cal)
print('XGBoost training complete.')


xgboost/best_cv_auc,▁
xgboost/best_param/colsample_bytree,▁
xgboost/best_param/learning_rate,▁
xgboost/best_param/max_depth,▁
xgboost/best_param/n_estimators,▁
xgboost/best_param/subsample,▁
xgboost/calibration_auc,▁
xgboost/best_cv_auc,0.70598
xgboost/best_param/colsample_bytree,0.8
xgboost/best_param/learning_rate,0.01
xgboost/best_param/max_depth,3


XGBoost training complete.


### 7) Train Logistic Regression and Log to W&B


In [22]:
from src.models.train import train_logistic

logreg_model = train_logistic(X_train_scaled, y_train, X_cal_scaled, y_cal)
print('Logistic Regression training complete.')


C:\Users\preet\Projects\major-project\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\preet\Projects\major-project\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


logistic/best_cv_auc,▁
logistic/best_param/C,▁
logistic/calibration_auc,▁
logistic/best_cv_auc,0.72022
logistic/best_param/C,0.01
logistic/best_param/penalty,l2
logistic/best_param/solver,lbfgs
logistic/calibration_auc,0.75268
logistic_calibration_auc,0.75268


Logistic Regression training complete.


### 8) Calibration-Set AUC Comparison


In [23]:
import pandas as pd
from sklearn.metrics import roc_auc_score

xgb_auc    = roc_auc_score(y_cal, xgb_model.predict_proba(X_cal_scaled)[:, 1])
logreg_auc = roc_auc_score(y_cal, logreg_model.predict_proba(X_cal_scaled)[:, 1])

comparison_df = pd.DataFrame({
    'model': ['XGBoost', 'LogisticRegression'],
    'calibration_auc': [round(xgb_auc, 4), round(logreg_auc, 4)],
})
print(comparison_df.to_string(index=False))


             model  calibration_auc
           XGBoost           0.7335
LogisticRegression           0.7527


### 9) Save Trained Models


In [ ]:
from src.models.train import save_model

xgb_path    = save_model(xgb_model,    'xgboost_model.pkl')
logreg_path = save_model(logreg_model, 'logistic_model.pkl')

print('Models saved:')
print(' ', xgb_path)
print(' ', logreg_path)
print('\nTraining complete. Run 04_calibration.ipynb next.')


artifact_path,C:\Users\preet\Proje...
